# Simulation 1 Cued Unguided Retrieval Figure

> Generate the Simulation 1 figure package for unguided retrieval with weak test-phase film cues.

This notebook crosses reminder condition with task encoding strength under unguided retrieval.
It holds weak VIT-style film cues constant and separates topline phase-total recall from serial-position and context diagnostics.

In [ ]:
import csv
import os
import warnings
from pathlib import Path

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
from jax import random
from jaxcmr.analyses.spc import fixed_pres_spc
from jaxcmr.helpers import find_project_root

from selective_interference_v2 import (
    PHASE_COLORS,
    Paradigm,
    add_phase_bands,
    apply_supplied_item_context_cue,
    batch_trial,
    configure_rates,
    light_to_dark_colors,
    load_fit_params,
    make_factory,
    make_is_emotional,
    make_is_target,
    prepare_sweep,
    save_figure,
    simulate_periodic_cued_free_recall,
    split_scales_for_cache,
    standard_remap,
    sweep_rngs,
)

warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]


In [ ]:
# --- sweep configuration ---
PROJECT_ROOT = ""
FIT_PATH = "results/fits/Dupertuys2026_eCMR_source_only_phi_no_shared_support_best_of_1.json"
FIGURE_DIR = ""
FIGURE_STR = ""
RNG_SEED = 0
SAVE_LEGACY_OUTPUTS = False
SAVE_DIAGNOSTIC_OUTPUTS = True

RASTER_N_TRIALS = 12
RASTER_MAX_OUTPUTS = 16
RASTER_SAMPLE_SEED_OFFSET = 1000

TASK_MCF_VALUES = [1.0, 2.0]
TASK_MCF_LABELS = {
    1.0: "Weak task encoding",
    2.0: "Strong task encoding",
}

PRIMARY_SOURCE_ORIENTATION = 0.0

FILM_CUE_REINSTATEMENT = 0.90
CUE_INTERVAL = 4
FIRST_CUE_AFTER = 0

N_FILM = 16
N_BREAK = 16
N_INTERFERENCE = 16
N_FILLER = 16
EXPERIMENT_COUNT = 5000
MAX_RECALL = 48

FILM_EMOTIONAL = True
INTERFERENCE_EMOTIONAL = False
CACHE_AFTER = "filler"

EXTRA_FIXED_SCALES = {
    "source_learning_baseline": 0.05,
    "primacy_scale": 0.1,
    "primacy_decay": 1.5,
}

BASE_FIXED_SCALES = {
    "break_drift_scale": 1.0,
    "break_mcf_scale": 1.0,
    "filler_drift_scale": 1.0,
    "filler_mcf_scale": 1.0,
}

REMINDER_CONDITIONS = {
    "No reminder": {
        "reminder_start_drift_scale": 0.0,
        "reminder_drift_scale": 0.0,
    },
    "With reminder": {
        "reminder_start_drift_scale": 1.0,
        "reminder_drift_scale": 1.0,
    },
}

RETRIEVAL_FIXED_SCALES = {
    "start_drift_scale": 0.0,
    "tau_scale": 1.0,
    "target_recall_drift_scale": 1.0,
    "rejected_recall_drift_scale": 1.0,
    "recall_drift_scale": 1.0,
}


In [ ]:
#| output: asis
if PROJECT_ROOT:
    project_root = Path(PROJECT_ROOT)
else:
    project_root = Path(find_project_root())

fit_path = project_root / FIT_PATH
figure_dir = project_root / FIGURE_DIR if FIGURE_DIR else None
task_mcf_values = np.asarray(TASK_MCF_VALUES, dtype=float)
extra_fixed_scales = {**EXTRA_FIXED_SCALES}
base_fixed_scales = {**BASE_FIXED_SCALES, **extra_fixed_scales}

params, n_subjects = load_fit_params(fit_path)
paradigm = Paradigm(
    n_film=N_FILM,
    n_break=N_BREAK,
    n_interference=N_INTERFERENCE,
    n_filler=N_FILLER,
    experiment_count=EXPERIMENT_COUNT,
    max_recall=MAX_RECALL,
)
factory = make_factory(
    is_emotional=make_is_emotional(
        paradigm,
        film_emotional=FILM_EMOTIONAL,
        interference_emotional=INTERFERENCE_EMOTIONAL,
    ),
    is_target=make_is_target(paradigm),
)
rng = random.PRNGKey(RNG_SEED)

Markdown(
    f"Loaded {n_subjects} fit parameter set(s). Running {len(REMINDER_CONDITIONS)} reminder conditions "
    f"and {len(task_mcf_values)} interference-strength values with a film cue before first recall "
    f"and {EXPERIMENT_COUNT} simulated trials per cell."
)


In [ ]:
def position_phase_labels(paradigm):
    return (
        ["film"] * paradigm.n_film
        + ["break"] * paradigm.n_break
        + ["task"] * paradigm.n_interference
        + ["filler"] * paradigm.n_filler
    )


def write_rows(path, fieldnames, rows):
    os.makedirs(path.parent, exist_ok=True)
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def phase_recall_totals(spc):
    totals = {}
    for phase in sorted(set(position_phase)):
        positions = [idx for idx, value in enumerate(position_phase) if value == phase]
        totals[phase] = float(np.sum(spc[positions]))
    return totals


def encode_items(model, items, method, collect_context=False):
    states = []
    for item in np.asarray(items, dtype=int):
        model = getattr(model, method)(jnp.asarray(int(item)))
        if collect_context:
            states.append(np.asarray(model.context.state))
    if collect_context:
        return model, np.asarray(states)
    return model


def normalize_rows(x):
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(norms, 1e-12)


def normalize_vector(x):
    return x / max(float(np.linalg.norm(x)), 1e-12)


position_phase = position_phase_labels(paradigm)
n_presented = len(position_phase)

In [ ]:
def cued_recall_trial(
    model,
    rng,
    film_items,
    cue_scale,
    max_recall,
    cue_interval,
    first_cue_after,
):
    recall_rng, initial_cue_rng = random.split(rng)
    model = model.start_retrieving()
    if first_cue_after == 0:
        cue_item = random.choice(initial_cue_rng, film_items)
        model = apply_supplied_item_context_cue(model, cue_item, cue_scale)
        scheduled_first_cue_after = cue_interval
    else:
        scheduled_first_cue_after = first_cue_after
    _, recalls, cue_attempts, cue_items = simulate_periodic_cued_free_recall(
        model,
        max_recall,
        recall_rng,
        film_items,
        cue_scale,
        cue_interval=cue_interval,
        first_cue_after=scheduled_first_cue_after,
    )
    return recalls, cue_attempts, cue_items


batched_cued_recall = batch_trial(cued_recall_trial, n_args=7, n_static=3)

prepared_cache = {}

for reminder_condition, reminder_scales in REMINDER_CONDITIONS.items():
    for task_scale in task_mcf_values:
        fixed_scales = {
            **base_fixed_scales,
            **reminder_scales,
            "interference_mcf_scale": float(task_scale),
        }
        pre_cache_scales, post_cache_scales = split_scales_for_cache(
            fixed_scales,
            CACHE_AFTER,
        )
        prepared = prepare_sweep(
            params,
            paradigm,
            factory,
            cache_after=CACHE_AFTER,
            **pre_cache_scales,
        )
        prepared_cache[(reminder_condition, float(task_scale))] = prepared


In [ ]:
cue_rows = []
cue_curves = {}
cue_recall_sequences = {}
cue_rng = random.PRNGKey(RNG_SEED + 1)

for reminder_condition, reminder_scales in REMINDER_CONDITIONS.items():
    for task_scale in task_mcf_values:
        fixed_scales = {
            **base_fixed_scales,
            **reminder_scales,
            "interference_mcf_scale": float(task_scale),
        }
        _, post_cache_scales = split_scales_for_cache(
            fixed_scales,
            CACHE_AFTER,
        )
        prepared = prepared_cache[(reminder_condition, float(task_scale))]
        ready_models = configure_rates(
            prepared.models,
            **post_cache_scales,
            **RETRIEVAL_FIXED_SCALES,
            film_source_start_drift_rate=float(PRIMARY_SOURCE_ORIENTATION),
        )
        rngs, cue_rng = sweep_rngs(
            cue_rng,
            prepared.n_subjects,
            prepared.experiment_count,
        )
        recalls, cue_attempts, cue_items = batched_cued_recall(
            ready_models,
            rngs,
            paradigm.film_items,
            FILM_CUE_REINSTATEMENT,
            paradigm.max_recall,
            CUE_INTERVAL,
            FIRST_CUE_AFTER,
        )
        raw_recalls = np.asarray(recalls).reshape(-1, recalls.shape[-1])
        remapped_recalls = np.asarray(
            standard_remap(
                raw_recalls,
                paradigm,
                show_break=True,
            )
        )
        cue_recall_sequences[(reminder_condition, float(task_scale))] = remapped_recalls
        spc = np.asarray(fixed_pres_spc(remapped_recalls, n_presented), dtype=float)
        totals = phase_recall_totals(spc)
        cue_curves[(reminder_condition, float(task_scale))] = spc
        for phase, total in totals.items():
            cue_rows.append({
                "reminder_condition": reminder_condition,
                "task_condition": TASK_MCF_LABELS[float(task_scale)],
                "task_mcf_scale": float(task_scale),
                "film_cue_reinstatement": FILM_CUE_REINSTATEMENT,
                "first_cue_after": FIRST_CUE_AFTER,
                "phase": phase,
                "recall_probability_mass": total,
            })


In [ ]:
def make_scalar_model(scales):
    one_subject_params = {key: value[:1] for key, value in params.items()}
    models = prepare_sweep(
        one_subject_params,
        paradigm,
        factory,
        cache_after="creation",
    ).models
    configured = configure_rates(models, **scales)
    return configured.replace(n_film=jnp.asarray([paradigm.n_film]))


def scalar_model(scales):
    model = make_scalar_model(scales)
    return jax_tree_first(model)


def jax_tree_first(model):
    from jax import tree_util

    return tree_util.tree_map(
        lambda x: x[0] if hasattr(x, "shape") and len(x.shape) > 0 and x.shape[0] == 1 else x,
        model,
    )


def trace_contexts(reminder_scales):
    fixed_scales = {**base_fixed_scales, **reminder_scales}
    model = scalar_model(fixed_scales)

    model, film_context = encode_items(
        model,
        paradigm.film_items,
        "experience_film",
        collect_context=True,
    )
    model = encode_items(model, paradigm.break_items, "experience_break")
    model = model.start_reminders()
    model = encode_items(model, paradigm.film_items, "remind")
    task_start_model = model
    model, task_context = encode_items(
        model,
        paradigm.interference_items,
        "experience_interference",
        collect_context=True,
    )
    return film_context, task_context, task_start_model


def film_context_support(task_start_model, film_context, task_scale):
    model = task_start_model.replace(interference_mcf_scale=jnp.asarray(task_scale))
    model = encode_items(model, paradigm.interference_items, "experience_interference")

    film_context_norm = normalize_rows(film_context)
    film_cue = normalize_vector(np.mean(film_context_norm, axis=0))
    temporal_support = np.asarray(model.mcf.probe(jnp.asarray(film_cue)))
    source_support = np.asarray(
        model.emotion_mcf.probe(task_start_model.emotion_context.state)
    )
    support = temporal_support + source_support

    film_slice = slice(0, paradigm.n_film)
    task_start = paradigm.n_film + paradigm.n_break
    task_slice = slice(task_start, task_start + paradigm.n_interference)
    film_temporal_support = float(np.sum(temporal_support[film_slice]))
    task_temporal_support = float(np.sum(temporal_support[task_slice]))
    film_source_support = float(np.sum(source_support[film_slice]))
    task_source_support = float(np.sum(source_support[task_slice]))
    film_support = float(np.sum(support[film_slice]))
    task_support = float(np.sum(support[task_slice]))
    return {
        "film_temporal_support": film_temporal_support,
        "task_temporal_support": task_temporal_support,
        "film_source_support": film_source_support,
        "task_source_support": task_source_support,
        "film_support": film_support,
        "task_support": task_support,
    }


In [ ]:
context_rows = []
support_rows = []
context_similarity = {}

for reminder_condition, reminder_scales in REMINDER_CONDITIONS.items():
    film_context, task_context, task_start_model = trace_contexts(reminder_scales)
    film_norm = normalize_rows(film_context)
    task_norm = normalize_rows(task_context)
    similarity = film_norm @ task_norm.T
    context_similarity[reminder_condition] = similarity

    for film_position in range(similarity.shape[0]):
        for task_position in range(similarity.shape[1]):
            context_rows.append({
                "reminder_condition": reminder_condition,
                "film_position": film_position + 1,
                "task_position": task_position + 1,
                "similarity": similarity[film_position, task_position],
            })

    for task_scale in task_mcf_values:
        support_summary = film_context_support(
            task_start_model,
            film_context,
            float(task_scale),
        )
        support_rows.append({
            "reminder_condition": reminder_condition,
            "task_condition": TASK_MCF_LABELS[float(task_scale)],
            "task_mcf_scale": float(task_scale),
            **support_summary,
        })


In [ ]:
PANEL_LETTER_FONTSIZE = 16
PANEL_TITLE_FONTSIZE = 13
STRUCTURAL_FONTSIZE = 11
SUPPORT_FONTSIZE = 10

REMINDER_LABELS = {
    "No reminder": "No reminder",
    "With reminder": "Film reminder",
}
REMINDER_SHORT_LABELS = {
    "No reminder": "no reminder",
    "With reminder": "reminder",
}
TASK_COLORS = {
    "Weak task encoding": "#9CA3AF",
    "Strong task encoding": PHASE_COLORS["task"],
}
REMINDER_LINESTYLES = {
    "No reminder": "--",
    "With reminder": "-",
}
REMINDER_MARKERS = {
    "No reminder": "o",
    "With reminder": "s",
}
BAR_REMINDER_LABELS = {
    "No reminder": "No reminder",
    "With reminder": "Film reminder",
}


def add_panel_label(axis, label, x=-0.16, y=1.12):
    axis.text(
        x,
        y,
        label,
        transform=axis.transAxes,
        fontsize=PANEL_LETTER_FONTSIZE,
        fontweight="bold",
        va="top",
        ha="left",
    )


def cue_total(reminder_condition, task_scale, phase):
    for row in cue_rows:
        if (
            row["reminder_condition"] == reminder_condition
            and np.isclose(row["task_mcf_scale"], task_scale)
            and row["phase"] == phase
        ):
            return row["recall_probability_mass"]
    return np.nan


def plot_2x2_film_recall(axis, value_fn, title=None, show_legend=True):
    x = np.arange(len(task_mcf_values))
    width = 0.34
    offsets = [-width / 2, width / 2]
    reminder_colors = {
        "No reminder": "#9CA3AF",
        "With reminder": PHASE_COLORS["reminder"],
    }
    all_values = []
    for offset, reminder_condition in zip(offsets, REMINDER_CONDITIONS):
        values = [
            value_fn(reminder_condition, float(task_scale))
            for task_scale in task_mcf_values
        ]
        all_values.extend(values)
        axis.bar(
            x + offset,
            values,
            width=width,
            color=reminder_colors[reminder_condition],
            edgecolor="white",
            linewidth=0.6,
            label=BAR_REMINDER_LABELS[reminder_condition],
            zorder=2,
        )
    upper = max(all_values) * 1.14 if all_values else 1.0
    upper = max(1.0, np.ceil(upper * 4) / 4)
    if title is not None:
        axis.set_title(title, fontsize=PANEL_TITLE_FONTSIZE, pad=8)
    axis.set_xticks(x)
    axis.set_xticklabels([TASK_MCF_LABELS[float(value)] for value in task_mcf_values])
    axis.set_xlabel("Interference strength", fontsize=STRUCTURAL_FONTSIZE)
    axis.set_ylabel("Film items recalled", fontsize=STRUCTURAL_FONTSIZE)
    axis.set_ylim(0, upper)
    axis.tick_params(labelsize=SUPPORT_FONTSIZE)
    axis.set_axisbelow(True)
    axis.grid(axis="y", color="#E7EBF0", linewidth=0.45, alpha=0.45, zorder=0)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if show_legend:
        axis.legend(
            frameon=False,
            fontsize=SUPPORT_FONTSIZE,
            loc="upper right",
            borderaxespad=0.3,
        )


def add_phase_boundaries(axis):
    boundaries = np.cumsum([
        paradigm.n_film,
        paradigm.n_break,
        paradigm.n_interference,
    ]) + 0.5
    for boundary in boundaries:
        axis.axvline(boundary, color="#7C8794", linewidth=0.7, alpha=0.45, zorder=1)


def add_reminder_marker(axis):
    reminder_x = paradigm.n_film + paradigm.n_break + 0.5
    axis.axvline(
        reminder_x,
        color=PHASE_COLORS["reminder"],
        linewidth=1.8,
        alpha=0.85,
        zorder=2,
    )
    axis.text(
        reminder_x - 1.05,
        0.62,
        "Film reminder",
        ha="center",
        va="center",
        fontsize=STRUCTURAL_FONTSIZE,
        fontstyle="italic",
        color=PHASE_COLORS["reminder"],
        rotation=90,
        rotation_mode="anchor",
        transform=axis.get_xaxis_transform(),
        clip_on=False,
        zorder=3,
    )


def plot_cue_spc_panel(axis, reminder_condition, label, show_reminder=False):
    add_phase_bands(axis, position_phase, fontsize=SUPPORT_FONTSIZE)
    add_phase_boundaries(axis)
    if show_reminder:
        add_reminder_marker(axis)
    positions = np.arange(1, n_presented + 1)
    for task_scale in task_mcf_values:
        task_label = TASK_MCF_LABELS[float(task_scale)]
        curve = cue_curves[(reminder_condition, float(task_scale))]
        axis.plot(
            positions,
            curve,
            color=TASK_COLORS[task_label],
            linewidth=1.7,
            label=task_label,
        )
    axis.set_title(REMINDER_LABELS[reminder_condition], fontsize=PANEL_TITLE_FONTSIZE, pad=34)
    axis.set_xlabel("Encoded position", fontsize=STRUCTURAL_FONTSIZE)
    axis.set_xlim(1, n_presented)
    axis.set_ylim(-0.025, 0.85)
    axis.tick_params(labelsize=SUPPORT_FONTSIZE)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    add_panel_label(axis, label)


In [ ]:
SEQUENCE_FIGURE_SUFFIX = "sequence_summary"
SEQUENCE_ROWS_SUFFIX = "sequence_rows"
PHASE_TOTALS_SUFFIX = "phase_totals_sequence_summary"
SEQUENCE_SELECTION_SUFFIX = "sequence_selection_summary"
SEQUENCE_SELECTION_RULE = "closest_to_condition_mean_film_count"

RASTER_PHASE_CODES = {
    "none": 0,
    "film": 1,
    "task": 2,
    "delay_filler": 3,
}
RASTER_PHASE_LABELS = {
    "none": "No output",
    "film": "Film",
    "task": "Task",
    "delay_filler": "Delay/filler",
}
RASTER_PHASE_COLORS = {
    "none": "#F7FAFF",
    "film": PHASE_COLORS["film"],
    "task": PHASE_COLORS["task"],
    "delay_filler": "#9CA3AF",
}


def recall_item_class(item_id):
    item_id = int(item_id)
    if item_id == 0:
        return "none"
    if 1 <= item_id <= paradigm.n_film:
        return "film"
    task_start = paradigm.n_film + paradigm.n_break + 1
    task_end = paradigm.n_film + paradigm.n_break + paradigm.n_interference
    if task_start <= item_id <= task_end:
        return "task"
    return "delay_filler"


def recall_phase_codes(recall_matrix):
    codes = np.zeros_like(recall_matrix, dtype=int)
    film_mask = (recall_matrix >= 1) & (recall_matrix <= paradigm.n_film)
    task_start = paradigm.n_film + paradigm.n_break + 1
    task_end = paradigm.n_film + paradigm.n_break + paradigm.n_interference
    task_mask = (recall_matrix >= task_start) & (recall_matrix <= task_end)
    other_mask = (recall_matrix != 0) & ~film_mask & ~task_mask
    codes[film_mask] = RASTER_PHASE_CODES["film"]
    codes[task_mask] = RASTER_PHASE_CODES["task"]
    codes[other_mask] = RASTER_PHASE_CODES["delay_filler"]
    return codes


def recall_phase_counts(recall_matrix):
    film_mask = (recall_matrix >= 1) & (recall_matrix <= paradigm.n_film)
    task_start = paradigm.n_film + paradigm.n_break + 1
    task_end = paradigm.n_film + paradigm.n_break + paradigm.n_interference
    task_mask = (recall_matrix >= task_start) & (recall_matrix <= task_end)
    nonzero_mask = recall_matrix != 0
    return {
        "film": np.sum(film_mask, axis=1),
        "task": np.sum(task_mask, axis=1),
        "delay_filler": np.sum(nonzero_mask & ~film_mask & ~task_mask, axis=1),
    }


def select_mean_film_trials(recalls, rng):
    counts = recall_phase_counts(recalls)
    film_counts = counts["film"]
    mean_film_count = float(np.mean(film_counts))
    distance = np.abs(film_counts - mean_film_count)
    tie_breakers = rng.random(recalls.shape[0])
    ranked_indices = np.lexsort((tie_breakers, distance))
    sample_size = min(RASTER_N_TRIALS, recalls.shape[0])
    selected_indices = np.sort(ranked_indices[:sample_size])
    rank_lookup = {int(trial_index): rank for rank, trial_index in enumerate(ranked_indices, start=1)}
    return selected_indices, rank_lookup, counts, mean_film_count, distance


def build_sequence_sample_rows():
    rng = np.random.default_rng(RNG_SEED + RASTER_SAMPLE_SEED_OFFSET)
    rows = []
    selection_rows = []
    samples = {}
    for reminder_condition in REMINDER_CONDITIONS:
        for task_scale in task_mcf_values:
            key = (reminder_condition, float(task_scale))
            recalls = cue_recall_sequences[key]
            selected_indices, rank_lookup, counts, mean_film_count, distance = select_mean_film_trials(recalls, rng)
            sampled = recalls[selected_indices, :RASTER_MAX_OUTPUTS]
            samples[key] = {
                "sample_indices": selected_indices,
                "recalls": sampled,
                "codes": recall_phase_codes(sampled),
            }
            for display_trial, trial_index in enumerate(selected_indices, start=1):
                trial_index = int(trial_index)
                film_count = int(counts["film"][trial_index])
                task_count = int(counts["task"][trial_index])
                delay_filler_count = int(counts["delay_filler"][trial_index])
                distance_from_mean = float(distance[trial_index])
                selection_rows.append({
                    "reminder_condition": reminder_condition,
                    "task_condition": TASK_MCF_LABELS[float(task_scale)],
                    "task_mcf_scale": float(task_scale),
                    "selection_rule": SEQUENCE_SELECTION_RULE,
                    "mean_film_count": mean_film_count,
                    "sampled_trial_index": trial_index,
                    "display_trial": display_trial,
                    "selection_rank": int(rank_lookup[trial_index]),
                    "film_count": film_count,
                    "task_count": task_count,
                    "delay_filler_count": delay_filler_count,
                    "distance_from_mean": distance_from_mean,
                })
                for output_position, item_id in enumerate(sampled[display_trial - 1], start=1):
                    item_class = recall_item_class(item_id)
                    rows.append({
                        "reminder_condition": reminder_condition,
                        "task_condition": TASK_MCF_LABELS[float(task_scale)],
                        "task_mcf_scale": float(task_scale),
                        "selection_rule": SEQUENCE_SELECTION_RULE,
                        "sampled_trial_index": trial_index,
                        "display_trial": display_trial,
                        "output_position": output_position,
                        "item_id": int(item_id),
                        "item_class": item_class,
                        "phase_code": RASTER_PHASE_CODES[item_class],
                    })
    return rows, selection_rows, samples


sequence_rows, sequence_selection_rows, sequence_samples = build_sequence_sample_rows()

if figure_dir is not None:
    write_rows(
        figure_dir / f"{FIGURE_STR}_{SEQUENCE_ROWS_SUFFIX}.csv",
        [
            "reminder_condition",
            "task_condition",
            "task_mcf_scale",
            "selection_rule",
            "sampled_trial_index",
            "display_trial",
            "output_position",
            "item_id",
            "item_class",
            "phase_code",
        ],
        sequence_rows,
    )
    write_rows(
        figure_dir / f"{FIGURE_STR}_{SEQUENCE_SELECTION_SUFFIX}.csv",
        [
            "reminder_condition",
            "task_condition",
            "task_mcf_scale",
            "selection_rule",
            "mean_film_count",
            "sampled_trial_index",
            "display_trial",
            "selection_rank",
            "film_count",
            "task_count",
            "delay_filler_count",
            "distance_from_mean",
        ],
        sequence_selection_rows,
    )
    write_rows(
        figure_dir / f"{FIGURE_STR}_{PHASE_TOTALS_SUFFIX}.csv",
        [
            "reminder_condition",
            "task_condition",
            "task_mcf_scale",
            "film_cue_reinstatement",
            "first_cue_after",
            "phase",
            "recall_probability_mass",
        ],
        cue_rows,
    )


def plot_sequence_raster(axis, reminder_condition, task_scale, show_xticks=False):
    sample = sequence_samples[(reminder_condition, float(task_scale))]["codes"]
    cmap = ListedColormap([
        RASTER_PHASE_COLORS["none"],
        RASTER_PHASE_COLORS["film"],
        RASTER_PHASE_COLORS["task"],
        RASTER_PHASE_COLORS["delay_filler"],
    ])
    norm = BoundaryNorm(np.arange(-0.5, 4.5, 1.0), cmap.N)
    axis.imshow(sample, cmap=cmap, norm=norm, aspect="auto", interpolation="none")
    axis.set_xlim(-0.5, RASTER_MAX_OUTPUTS - 0.5)
    axis.set_ylim(sample.shape[0] - 0.5, -0.5)
    if show_xticks:
        tick_positions = [0, 7, min(15, RASTER_MAX_OUTPUTS - 1)]
        axis.set_xticks(tick_positions)
        axis.set_xticklabels([str(position + 1) for position in tick_positions])
        axis.tick_params(axis="x", labelsize=SUPPORT_FONTSIZE, length=2.5, pad=2)
    else:
        axis.set_xticks([])
    axis.set_yticks([])
    axis.set_xticks(np.arange(-0.5, RASTER_MAX_OUTPUTS, 1), minor=True)
    axis.set_yticks(np.arange(-0.5, sample.shape[0], 1), minor=True)
    axis.grid(which="minor", color="white", linewidth=0.5)
    axis.tick_params(which="minor", bottom=False, left=False)
    axis.set_facecolor("#FAFBFC")
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_color("#3D4B5C")
        spine.set_linewidth(1.2)


def plot_2x2_phase_recall(axis, phase, title, color, pale_color, show_legend=False, ylim=None):
    x = np.arange(len(task_mcf_values))
    width = 0.34
    offsets = [-width / 2, width / 2]
    all_values = []
    for offset, reminder_condition in zip(offsets, REMINDER_CONDITIONS):
        values = [
            cue_total(reminder_condition, float(task_scale), phase)
            for task_scale in task_mcf_values
        ]
        all_values.extend(values)
        is_reminder = reminder_condition == "With reminder"
        axis.bar(
            x + offset,
            values,
            width=width,
            color=color if is_reminder else pale_color,
            edgecolor=color,
            linewidth=1.1,
            label=BAR_REMINDER_LABELS[reminder_condition],
            zorder=2,
        )
    axis.set_title(title, fontsize=PANEL_TITLE_FONTSIZE, pad=8)
    axis.set_xticks(x)
    axis.set_xticklabels([TASK_MCF_LABELS[float(value)] for value in task_mcf_values])
    axis.set_ylabel("Mean film items recalled", fontsize=STRUCTURAL_FONTSIZE)
    if ylim is None:
        upper = max(all_values) * 1.14 if all_values else 1.0
        upper = max(1.0, np.ceil(upper * 4) / 4)
        axis.set_ylim(0, upper)
    else:
        axis.set_ylim(ylim)
    axis.tick_params(labelsize=SUPPORT_FONTSIZE)
    axis.set_axisbelow(True)
    axis.grid(axis="y", color="#E7EBF0", linewidth=0.45, alpha=0.45, zorder=0)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if show_legend:
        legend_handles = [
            Patch(facecolor=pale_color, edgecolor=color, linewidth=1.1, label="No reminder"),
            Patch(facecolor=color, edgecolor=color, linewidth=1.1, label="Film reminder"),
        ]
        axis.legend(
            handles=legend_handles,
            frameon=False,
            fontsize=STRUCTURAL_FONTSIZE,
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
            borderaxespad=0.0,
        )


film_values = [row["recall_probability_mass"] for row in cue_rows if row["phase"] == "film"]
film_bar_ylim = (0, max(1.0, np.ceil(max(film_values) * 1.14 * 4) / 4))

fig = plt.figure(figsize=(10.8, 6.8))
gs = fig.add_gridspec(
    2,
    4,
    height_ratios=[1.15, 0.95],
    hspace=0.55,
    wspace=0.30,
    left=0.18,
    right=0.985,
    top=0.875,
    bottom=0.090,
)

panel_a_gs = gs[0, :].subgridspec(2, 2, hspace=0.20, wspace=0.12)
panel_a_axes = {}
for row_index, task_scale in enumerate(task_mcf_values):
    for col_index, reminder_condition in enumerate(REMINDER_CONDITIONS):
        axis = fig.add_subplot(panel_a_gs[row_index, col_index])
        show_xticks = row_index == len(task_mcf_values) - 1
        plot_sequence_raster(axis, reminder_condition, float(task_scale), show_xticks=show_xticks)
        if row_index == 0:
            axis.set_title(
                BAR_REMINDER_LABELS[reminder_condition],
                fontsize=STRUCTURAL_FONTSIZE,
                pad=8,
            )
        if col_index == 0:
            axis.set_ylabel(
                TASK_MCF_LABELS[float(task_scale)],
                fontsize=STRUCTURAL_FONTSIZE,
                labelpad=18,
                rotation=0,
                ha="right",
                va="center",
            )
        if row_index == len(task_mcf_values) - 1:
            axis.set_xlabel("output position", fontsize=SUPPORT_FONTSIZE, labelpad=5)
        panel_a_axes[(reminder_condition, float(task_scale))] = axis

legend_handles = [
    Patch(facecolor=RASTER_PHASE_COLORS["film"], edgecolor="none", label=RASTER_PHASE_LABELS["film"]),
    Patch(facecolor=RASTER_PHASE_COLORS["task"], edgecolor="none", label=RASTER_PHASE_LABELS["task"]),
    Patch(facecolor=RASTER_PHASE_COLORS["delay_filler"], edgecolor="none", label=RASTER_PHASE_LABELS["delay_filler"]),
    Patch(facecolor=RASTER_PHASE_COLORS["none"], edgecolor="#C7D0DA", label=RASTER_PHASE_LABELS["none"]),
]
panel_a_left = min(axis.get_position().x0 for axis in panel_a_axes.values())
panel_a_right = max(axis.get_position().x1 for axis in panel_a_axes.values())
panel_a_center = (panel_a_left + panel_a_right) / 2
fig.text(
    panel_a_center,
    0.974,
    "Example recall sequences",
    fontsize=PANEL_TITLE_FONTSIZE,
    ha="center",
    va="top",
)
fig.legend(
    handles=legend_handles,
    frameon=False,
    fontsize=SUPPORT_FONTSIZE,
    loc="upper center",
    bbox_to_anchor=(panel_a_center, 0.944),
    ncol=4,
    columnspacing=1.2,
    handlelength=1.3,
)

fig.text(
    panel_a_left - 0.09,
    0.965,
    "A",
    fontsize=PANEL_LETTER_FONTSIZE,
    fontweight="bold",
    ha="left",
    va="top",
)

axis_b = fig.add_subplot(gs[1, 1:3])
plot_2x2_phase_recall(
    axis_b,
    "film",
    "Film output across all trials",
    PHASE_COLORS["film"],
    "#EAF2FF",
    show_legend=True,
    ylim=film_bar_ylim,
)
add_panel_label(axis_b, "B", x=-0.24, y=1.17)

save_figure(str(figure_dir) if figure_dir is not None else "", FIGURE_STR, SEQUENCE_FIGURE_SUFFIX)



In [ ]:
if SAVE_LEGACY_OUTPUTS:
    fig, axis = plt.subplots(figsize=(4.8, 3.6))
    plot_2x2_film_recall(
        axis,
        lambda reminder_condition, task_scale: cue_total(
            reminder_condition,
            task_scale,
            "film",
        ),
        title=None,
    )
    save_figure(str(figure_dir) if figure_dir is not None else "", FIGURE_STR, "2x2_film_recall_grouped_by_task_cue_090_firstcue0")
else:
    print("Skipping legacy 2x2 figure save.")


In [ ]:
if SAVE_DIAGNOSTIC_OUTPUTS:
    fig = plt.figure(figsize=(13, 3.55))
    gs = fig.add_gridspec(1, 2, wspace=0.38)

    axis_a = fig.add_subplot(gs[0, 0])
    axis_b = fig.add_subplot(gs[0, 1], sharey=axis_a)
    plot_cue_spc_panel(axis_a, "No reminder", "A")
    plot_cue_spc_panel(axis_b, "With reminder", "B", show_reminder=True)
    axis_a.set_ylabel("Recall probability", fontsize=STRUCTURAL_FONTSIZE)
    axis_b.set_ylabel("Recall probability", fontsize=STRUCTURAL_FONTSIZE)
    axis_b.tick_params(labelleft=True)
    axis_b.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        fontsize=SUPPORT_FONTSIZE,
    )
    save_figure(str(figure_dir) if figure_dir is not None else "", FIGURE_STR, "recall_distributions")

    fig = plt.figure(figsize=(13, 3.45))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.02, 1.0], wspace=0.40)

    heatmap_gs = gs[0, 0].subgridspec(1, 3, width_ratios=[1, 1, 0.055], wspace=0.16)
    axis_h1 = fig.add_subplot(heatmap_gs[0, 0])
    axis_h2 = fig.add_subplot(heatmap_gs[0, 1], sharey=axis_h1)
    cbar_axis = fig.add_subplot(heatmap_gs[0, 2])
    heatmap_axes = [(axis_h1, "No reminder"), (axis_h2, "With reminder")]
    similarity_max = max(float(np.max(value)) for value in context_similarity.values())
    heatmap_image = None
    for axis, reminder_condition in heatmap_axes:
        heatmap_image = axis.imshow(
            context_similarity[reminder_condition],
            vmin=0,
            vmax=similarity_max,
            cmap="viridis",
            aspect="auto",
            interpolation="none",
            origin="lower",
        )
        axis.set_title(REMINDER_LABELS[reminder_condition], fontsize=STRUCTURAL_FONTSIZE, pad=8)
        axis.set_xlabel("Encoded task position", fontsize=STRUCTURAL_FONTSIZE)
        axis.set_xticks([0, paradigm.n_interference - 1])
        axis.set_xticklabels(["1", str(paradigm.n_interference)])
        axis.set_yticks([0, paradigm.n_film - 1])
        axis.set_yticklabels(["1", str(paradigm.n_film)])
        axis.tick_params(labelsize=SUPPORT_FONTSIZE)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)
    axis_h1.set_ylabel("Encoded film position", fontsize=STRUCTURAL_FONTSIZE)
    axis_h2.tick_params(labelleft=False)
    if heatmap_image is not None:
        cbar = fig.colorbar(heatmap_image, cax=cbar_axis)
        cbar.ax.tick_params(labelsize=SUPPORT_FONTSIZE)
    axis_h1.text(
        1.08,
        1.16,
        "Similarity between film and task contexts",
        transform=axis_h1.transAxes,
        ha="center",
        va="bottom",
        fontsize=PANEL_TITLE_FONTSIZE,
    )
    add_panel_label(axis_h1, "A", x=-0.35)

    support_gs = gs[0, 1].subgridspec(1, 2, wspace=0.22)
    axis_s1 = fig.add_subplot(support_gs[0, 0])
    axis_s2 = fig.add_subplot(support_gs[0, 1], sharey=axis_s1)
    support_axes = [(axis_s1, task_mcf_values[0]), (axis_s2, task_mcf_values[1])]
    item_classes = [
        ("film_support", "Film", PHASE_COLORS["film"]),
        ("task_support", "Task", PHASE_COLORS["task"]),
    ]
    support_max = max(
        max(row["film_support"], row["task_support"])
        for row in support_rows
    )

    def support_value(reminder_condition, task_scale, support_key):
        for row in support_rows:
            if (
                row["reminder_condition"] == reminder_condition
                and np.isclose(row["task_mcf_scale"], task_scale)
            ):
                return row[support_key]
        return np.nan

    x = np.arange(len(REMINDER_CONDITIONS))
    width = 0.34
    offsets = [-width / 2, width / 2]
    for axis, task_scale in support_axes:
        task_label = TASK_MCF_LABELS[float(task_scale)]
        for offset, (support_key, item_label, color) in zip(offsets, item_classes):
            values = [
                support_value(reminder_condition, float(task_scale), support_key)
                for reminder_condition in REMINDER_CONDITIONS
            ]
            axis.bar(
                x + offset,
                values,
                width=width,
                color=color,
                label=item_label,
                zorder=2,
            )
        axis.set_title(task_label, fontsize=STRUCTURAL_FONTSIZE, pad=8)
        axis.set_xticks(x)
        axis.set_xticklabels([BAR_REMINDER_LABELS[value] for value in REMINDER_CONDITIONS])
        axis.set_ylim(0, support_max * 1.18)
        axis.tick_params(labelsize=SUPPORT_FONTSIZE)
        axis.grid(axis="y", color="#D6DBE1", linewidth=0.7, alpha=0.7, zorder=0)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)
    axis_s1.set_ylabel("Support from film context", fontsize=STRUCTURAL_FONTSIZE)
    axis_s2.tick_params(labelleft=False)
    axis_s2.legend(
        frameon=False,
        fontsize=SUPPORT_FONTSIZE,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
    )
    axis_s1.text(
        1.08,
        1.16,
        "Items supported by film context",
        transform=axis_s1.transAxes,
        ha="center",
        va="bottom",
        fontsize=PANEL_TITLE_FONTSIZE,
    )
    add_panel_label(axis_s1, "B", x=-0.35)

    save_figure(str(figure_dir) if figure_dir is not None else "", FIGURE_STR, "context_interference")
else:
    print("Skipping diagnostic mechanism figure save.")


In [ ]:
print("Film cue before first recall")
for reminder_condition in REMINDER_CONDITIONS:
    for task_scale in task_mcf_values:
        film_total = cue_total(reminder_condition, float(task_scale), "film")
        task_total = cue_total(reminder_condition, float(task_scale), "task")
        print(
            f"  {reminder_condition}, {TASK_MCF_LABELS[float(task_scale)]}: "
            f"film={film_total:.3f}, task={task_total:.3f}"
        )
